# TTMPC Credit Risk — Master Production Notebook

This notebook produces the trained credit risk model that will be handed to the development team for integration into the TTMPC Loan Management System.

## Inputs
- **`Loan_Ledger.xlsx`** — Multi-sheet Excel file with each member's loan history
- **`MembersProfile.csv`** — Member demographic and income data

## Outputs (the handoff package)
- **`ttmpc_credit_risk_model.pkl`** — Trained Random Forest classifier
- **`model_metadata.json`** — Machine-readable feature specification

## Process (CRISP-DM Phase 3–4)
1. **Setup** — mount Drive, configure paths
2. **Phase 3 — Data Preparation** — extract, clean, link, build master matrix
3. **Phase 4 — Modeling** — feature engineering, train, save, verify
4. **Handoff verification**

## Notes
This is the **production** notebook. Exploratory analysis, statistical validation, visualizations, and operational audits are kept in a separate methodology notebook for thesis documentation. They are not needed to produce or use the model.

Re-running this notebook is fast: after the first run, intermediate data is cached in Google Drive. Only the modeling step re-runs unless you delete the cache.


## 1. Setup

Mount Google Drive and configure file paths. Edit `PROJECT_DIR` below to match your folder structure in Drive.


In [ ]:
import pandas as pd
import numpy as np
import uuid
import re
import json
import joblib
from pathlib import Path
from datetime import datetime
from google.colab import drive

# --- Mount Google Drive ---
drive.mount('/content/drive', force_remount=False)

# --- Configure paths (EDIT THIS to match your Drive folder) ---
PROJECT_DIR = Path('/content/drive/MyDrive/TTMPC_Credit_Risk')

# Input files
LEDGER_FILE  = PROJECT_DIR / 'Loan_Ledger.xlsx'
PROFILE_FILE = PROJECT_DIR / 'MembersProfile.csv'

# Output directories
OUTPUT_DIR = PROJECT_DIR / 'outputs'
CACHE_DIR  = PROJECT_DIR / 'cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Output files (the handoff package)
MODEL_FILE    = OUTPUT_DIR / 'ttmpc_credit_risk_model.pkl'
METADATA_FILE = OUTPUT_DIR / 'model_metadata.json'

print("Configuration:")
print(f"  Project dir : {PROJECT_DIR}")
print(f"  Ledger      : {LEDGER_FILE}")
print(f"  Profiles    : {PROFILE_FILE}")
print(f"  Outputs to  : {OUTPUT_DIR}")


## 2. Phase 3 — Data Preparation

### 2.1 Extract loans and payments from the Excel ledger

The Excel file has one sheet per member, with two loan sections per sheet (Consolidated in columns 0–8, Emergency in columns 10–18). This step parses every sheet and produces three flat tables: members, loans, and payments.

**Performance note:** the first run reads the raw Excel file (slow). Subsequent runs load from a parquet cache in your Drive (fast). To force re-extraction, delete the files in `CACHE_DIR`.


In [ ]:
def extract_data(df, db_uuid, loan_type, state):
    """
    Parse one loan section (Consolidated or Emergency) from a member's sheet.

    Appends discovered loans and payments to the lists in `state`. Mutates `state` counters.
    Behavior preserved exactly from the original working notebook.
    """
    if df.empty or df.shape[1] < 2:
        return
    df_str = df.astype(str)

    # Find rows where column index 1 contains 'Amount' or 'Amount of Loan' — these are loan headers
    headers = df[
        df.iloc[:, 1].astype(str).str.contains(r"^Amount$", na=False, case=False) |
        df.iloc[:, 1].astype(str).str.contains("Amount of Loan", na=False, case=False)
    ].index.tolist()

    for start_idx in headers:
        loan_uuid = str(uuid.uuid4())
        loan_code = f"TTMPCL-{state['loan_counter']:03d}"
        state['loan_counter'] += 1

        # Identify column positions from the header row text
        h_row = df.iloc[start_idx].astype(str).str.lower().reset_index(drop=True)
        def get_col(k):
            match = h_row[h_row.str.contains(k, na=False)].index.tolist()
            return match[0] if match else None

        col_date = get_col('date')      if get_col('date')      is not None else 0
        col_amt  = get_col('amount')    if get_col('amount')    is not None else 1
        col_bal  = get_col('balance')
        col_prin = get_col('principal') or get_col('repayment')
        col_or   = get_col('or/cdv')    or get_col('or #')

        if col_prin is None or col_bal is None:
            continue

        # Loan header values are on the row AFTER the header label
        raw_val  = str(df.iloc[start_idx + 1, col_amt]).replace(',', '').strip()
        amt      = pd.to_numeric(raw_val, errors='coerce')
        app_date = pd.to_datetime(df.iloc[start_idx + 1, col_date], errors='coerce')

        state['loans_table'].append({
            "LoanID": loan_uuid, "LoanCode": loan_code, "MemberUUID": db_uuid,
            "LoanAmount": amt, "ApplicationDate": app_date, "LoanType": loan_type
        })

        # Walk down the payment rows until balance reaches zero
        curr = start_idx + 1
        while curr < len(df) and pd.notna(df.iloc[curr, col_date]):
            p_val   = pd.to_numeric(str(df.iloc[curr, col_prin]).replace(',', ''), errors='coerce')
            or_val  = str(df.iloc[curr, col_or]).strip() if col_or is not None else ""
            row_txt = "".join(df_str.iloc[curr].values).upper()
            has_overpayment = "OVERPAYMENT" in row_txt

            if p_val and p_val > 0:
                p_date = pd.to_datetime(df.iloc[curr, col_date], errors='coerce')
                state['payments_table'].append({
                    "PaymentID":   str(uuid.uuid4()),
                    "PaymentCode": f"TTMPCP-{state['payment_counter']:03d}",
                    "LoanID":      loan_uuid,
                    "AmountPaid":  p_val,
                    "PaymentDate": p_date,
                    "OR_CDV_No":   or_val,
                    "Ledger_Note_Overpayment": has_overpayment
                })
                state['payment_counter'] += 1

            bal = pd.to_numeric(str(df.iloc[curr, col_bal]).replace(',', ''), errors='coerce')
            if pd.notna(bal) and bal <= 0.05:
                break
            curr += 1


# --- Cache check ---
cache_files = {
    'members':  CACHE_DIR / 'members.parquet',
    'loans':    CACHE_DIR / 'loans.parquet',
    'payments': CACHE_DIR / 'payments.parquet',
}

if all(f.exists() for f in cache_files.values()):
    print("📦 Loading from cache...")
    df_members  = pd.read_parquet(cache_files['members'])
    df_loans    = pd.read_parquet(cache_files['loans'])
    df_payments = pd.read_parquet(cache_files['payments'])
else:
    print("📂 Reading raw Excel ledger (first run)...")
    xl = pd.ExcelFile(LEDGER_FILE)

    # Build member map (sorted alphabetically by last name)
    sorting = []
    for sheet in xl.sheet_names:
        sort_name = sheet.split(',')[0].strip().upper() if ',' in sheet else sheet.strip().upper()
        sorting.append({'sheet_name': sheet, 'sort_name': sort_name})
    sorting.sort(key=lambda x: x['sort_name'])
    member_map = {
        s['sheet_name']: {'db_uuid': str(uuid.uuid4()), 'human_code': f"TTMPCM-{i:03d}"}
        for i, s in enumerate(sorting, start=1)
    }

    # Run extraction
    state = {'loans_table': [], 'payments_table': [], 'loan_counter': 1, 'payment_counter': 1}
    members_table = []
    for sheet in xl.sheet_names:
        if sheet not in member_map:
            continue
        m = member_map[sheet]
        last  = sheet.split(',')[0].strip() if ',' in sheet else sheet
        first = sheet.split(',')[1].strip() if ',' in sheet else ""
        members_table.append({
            "MemberUUID": m['db_uuid'], "MemberID": m['human_code'],
            "LastName": last, "FirstName": first
        })
        sheet_df = xl.parse(sheet)
        extract_data(sheet_df.iloc[:, 0:9],   m['db_uuid'], "Consolidated", state)
        extract_data(sheet_df.iloc[:, 10:19], m['db_uuid'], "Emergency",    state)

    df_members  = pd.DataFrame(members_table)
    df_loans    = pd.DataFrame(state['loans_table'])
    df_payments = pd.DataFrame(state['payments_table'])

    # Standardize member name strings (consolidated from old standardization cells)
    def clean_name(x):
        if pd.isna(x):
            return x
        return " ".join(re.sub(r"[^a-zA-Z\s\-\']", "", str(x)).split()).title()
    df_members['LastName']  = df_members['LastName'].apply(clean_name)
    df_members['FirstName'] = df_members['FirstName'].apply(clean_name)

    # Cache
    df_members.to_parquet(cache_files['members'])
    df_loans.to_parquet(cache_files['loans'])
    df_payments.to_parquet(cache_files['payments'])
    print(f"💾 Cached: {len(df_members)} members, {len(df_loans)} loans, {len(df_payments)} payments")

print(f"\n✅ Loaded: {len(df_members)} members | {len(df_loans)} loans | {len(df_payments)} payments")


### 2.2 Load profiles and map occupations to stability tiers

Occupations are bucketed into four risk tiers based on employment stability:
- **Public_Sector_Institutional** (score 4) — most stable
- **Private_Professional_Skilled** (score 3)
- **Service_Support** (score 2)
- **Entrepreneurial_Informal** (score 1) — most volatile
- **Unclassified_High_Risk** (score 2.0) — fallback for unknown occupations

The mapping tables here are the **single source of truth** for occupation handling. Any new occupation that appears in production data and isn't in this map will fall through to `Unclassified_High_Risk`.


In [ ]:
# =============================================================
# OCCUPATION TIER SYSTEM — single source of truth
# =============================================================
OCCUPATION_TIER_MAP = {
    # Public Sector Institutional
    'Teaching':                              'Public_Sector_Institutional',
    'Retired Teacher':                       'Public_Sector_Institutional',
    'Government Employee':                   'Public_Sector_Institutional',
    'Adas Iii':                              'Public_Sector_Institutional',
    'Local Treasury Operations Offices Ii':  'Public_Sector_Institutional',
    'Mpiw Employee':                         'Public_Sector_Institutional',
    'Admin Officer V':                       'Public_Sector_Institutional',
    'Adas Ii':                               'Public_Sector_Institutional',
    'Ada':                                   'Public_Sector_Institutional',
    'Social Welfare Assistant':              'Public_Sector_Institutional',
    'Senior Fire Inspector':                 'Public_Sector_Institutional',
    'Encoder':                               'Public_Sector_Institutional',
    'Assistant Pharmacist':                  'Public_Sector_Institutional',
    'Dietitian':                             'Public_Sector_Institutional',
    'Sb Member':                             'Public_Sector_Institutional',
    # Private Professional/Skilled
    'Seafarer':                              'Private_Professional_Skilled',
    'Automotive Technician':                 'Private_Professional_Skilled',
    'Beautician':                            'Private_Professional_Skilled',
    'Caregiver':                             'Private_Professional_Skilled',
    'Assistant Embalmer':                    'Private_Professional_Skilled',
    # Service / Support
    'Cashier':                               'Service_Support',
    'Store Clerk':                           'Service_Support',
    'Security Guard':                        'Service_Support',
    'Asst. Station Manager':                 'Service_Support',
    'Barber':                                'Service_Support',
    # Entrepreneurial / Informal
    'Business':                              'Entrepreneurial_Informal',
    'Vendor':                                'Entrepreneurial_Informal',
    'Entrepreneur':                          'Entrepreneurial_Informal',
    'Farmer':                                'Entrepreneurial_Informal',
    'Rice Dealer':                           'Entrepreneurial_Informal',
    'Small Business':                        'Entrepreneurial_Informal',
    # Fallback
    'Unknown':                               'Unclassified_High_Risk',
}

TIER_SCORE_MAP = {
    'Public_Sector_Institutional': 4,
    'Private_Professional_Skilled': 3,
    'Service_Support': 2,
    'Entrepreneurial_Informal': 1,
    'Unclassified_High_Risk': 2.0,  # conservative buffer for unknown occupations
}


def clean_occupation(occ):
    """Normalize occupation strings (Title Case, common typo fix)."""
    if pd.isna(occ) or str(occ).lower() == 'nan' or str(occ).strip() == '':
        return 'Unknown'
    occ = str(occ).strip().title()
    if 'Automative' in occ:
        occ = 'Automotive Technician'
    return occ


# --- Load and clean profiles ---
df_profiles = pd.read_csv(PROFILE_FILE)

df_profiles['Occupation_Cleaned'] = df_profiles['Occupation'].apply(clean_occupation)
df_profiles['occ_tier']           = df_profiles['Occupation_Cleaned'].map(OCCUPATION_TIER_MAP).fillna('Unclassified_High_Risk')
df_profiles['Stability_Score']    = df_profiles['occ_tier'].map(TIER_SCORE_MAP)

# Clean income (strip commas, coerce to numeric)
df_profiles['AnnualIncome'] = pd.to_numeric(
    df_profiles['AnnualIncome'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce'
)

print(f"✅ Loaded {len(df_profiles)} profiles")
print(f"\nOccupation tier distribution:")
print(df_profiles['occ_tier'].value_counts().to_string())
miss = df_profiles['AnnualIncome'].isna().sum()
print(f"\nIncome missing: {miss}/{len(df_profiles)} ({miss/len(df_profiles)*100:.1f}%)")


### 2.3 Link members across systems via name matching

The ledger and the profile CSV use different ID systems with no shared key, so we bridge them with a normalized name key (`LASTNAME_FIRSTNAME`). The resulting `MasterUUID` becomes the system-wide identifier.

**Known limitation:** ~17 members appear in the ledger but not in the profile CSV. These are reported below for manual review but won't enter the training set.


In [ ]:
# Build normalized match keys
df_members['match_key']  = (df_members['LastName'].str.upper()  + '_' + df_members['FirstName'].str.upper()).str.strip()
df_profiles['match_key'] = (df_profiles['LastName'].str.upper() + '_' + df_profiles['FirstName'].str.upper()).str.strip()

# Identify the profile's primary ID column (varies in source data)
profile_id_col = 'membership_number_id' if 'membership_number_id' in df_profiles.columns else 'MemberUUID'

# Build name -> MasterUUID lookup
name_to_master = df_profiles.set_index('match_key')[profile_id_col].to_dict()

# Apply to members and propagate to loans
df_members['MasterUUID'] = df_members['match_key'].map(name_to_master)
df_loans['MasterUUID']   = df_loans['MemberUUID'].map(df_members.set_index('MemberUUID')['MasterUUID'])

# Report match rate
matched = df_members['MasterUUID'].notna().sum()
total   = len(df_members)
print(f"✅ Matched {matched}/{total} members ({matched/total*100:.1f}%)")

unmatched = df_members[df_members['MasterUUID'].isna()]
if not unmatched.empty:
    print(f"\n⚠️ {len(unmatched)} unmatched members (in ledger, not in profile CSV):")
    print(unmatched[['LastName', 'FirstName']].to_string(index=False))


### 2.4 Clean payment ledger and detect advance payers

Some payments occur 1–30 days *before* the loan application date. This is a legitimate "advance payer" pattern in the cooperative (members making early deposits), not a data error. We distinguish those from real date errors (more than 30 days before application).

We also drop "projection rows" — placeholder payments without an OR/CDV receipt number, which represent future planned payments rather than actual ones.


In [ ]:
def clean_ledger_data(df_loans, df_payments):
    """
    Clean payments and classify the timing of each payment relative to its loan.

    Returns DataFrame with new columns:
      - delta_days:            payment date minus application date
      - is_advance_payer:      True if delta_days ∈ [-30, -1]  (legitimate early payment)
      - is_date_error:         True if delta_days < -30        (suspect data)
      - manual_note_overpayment: passed through from extraction
    """
    df_l = df_loans.copy()
    df_p = df_payments.copy()
    df_l['ApplicationDate'] = pd.to_datetime(df_l['ApplicationDate'])
    df_p['PaymentDate']     = pd.to_datetime(df_p['PaymentDate'])

    # Drop placeholder payments (no OR/CDV receipt = future projection, not real payment)
    if 'OR_CDV_No' in df_p.columns:
        df_p['OR_CDV_No'] = df_p['OR_CDV_No'].astype(str).str.strip().replace(['', 'nan', 'None'], np.nan)
        before = len(df_p)
        df_p = df_p.dropna(subset=['OR_CDV_No']).copy()
        print(f"   Dropped {before - len(df_p)} projection rows.")

    # Bring loan dates and MasterUUID into the payments
    loans_lookup = df_l.loc[:, ~df_l.columns.duplicated()][['LoanID', 'ApplicationDate', 'MasterUUID']]
    merged = df_p.merge(loans_lookup, on='LoanID', how='left')
    merged['delta_days'] = (merged['PaymentDate'] - merged['ApplicationDate']).dt.days

    # Classify payment timing
    conditions = [
        (merged['delta_days'] >= -30) & (merged['delta_days'] <= -1),  # Advance payer
        (merged['delta_days'] < -30),                                   # Date error
    ]
    merged['is_advance_payer'] = np.select(conditions, [True, False], default=False)
    merged['is_date_error']    = np.select(conditions, [False, True], default=False)
    merged['manual_note_overpayment'] = merged.get('Ledger_Note_Overpayment', False)

    return merged


df_payments_clean = clean_ledger_data(df_loans, df_payments)
print(f"\n✅ Clean ledger: {len(df_payments_clean)} payments")
print(f"   Advance payers : {df_payments_clean['is_advance_payer'].sum()}")
print(f"   Date errors    : {df_payments_clean['is_date_error'].sum()}")


### 2.5 Build the master analytical matrix

This produces **one row per loan** with everything needed for modeling: loan info, payment aggregates, member profile data, target label, and the Repayment Stress Index.

**Target definition:**
```
Target_Risk = 1  if  (RemainingPrincipal > 15% of LoanAmount)  AND  (Advance_Payment_Count == 0)
Target_Risk = 0  otherwise
```

This captures the operational risk pattern: a borrower who has paid down less than 85% of principal *and* shows no advance-payment behavior is flagged as high risk.


In [ ]:
def build_master_matrix(df_profiles, df_payments_clean, df_loans, profile_id_col):
    """
    Build the master analytical matrix: one row per loan, ready for modeling.
    """
    df_p = df_profiles.loc[:, ~df_profiles.columns.duplicated()].copy()
    df_l = df_loans.loc[:, ~df_loans.columns.duplicated()].copy()
    df_t = df_payments_clean.loc[:, ~df_payments_clean.columns.duplicated()].copy()

    # 1. Aggregate payments to loan grain
    rollups = df_t.groupby('LoanID').agg(
        Actual_Total_Paid=('AmountPaid', 'sum'),
        Advance_Payment_Count=('is_advance_payer', lambda x: int((x == True).sum())),
        Overpayment_Note_Count=('manual_note_overpayment', lambda x: int((x == True).sum())),
    ).reset_index()

    # 2. Merge aggregates into loans
    matrix = df_l.merge(rollups, on='LoanID', how='left').fillna({
        'Actual_Total_Paid': 0,
        'Advance_Payment_Count': 0,
        'Overpayment_Note_Count': 0,
    })

    # 3. Bring in income from profiles
    income_map = df_p[[profile_id_col, 'AnnualIncome']].rename(
        columns={profile_id_col: 'MasterUUID'}
    ).drop_duplicates('MasterUUID')
    matrix = matrix.merge(income_map, on='MasterUUID', how='left')

    # 4. Compute target: high risk if >15% principal unpaid AND no advance payments
    matrix['Target_Risk'] = np.where(
        (matrix['LoanAmount'] - matrix['Actual_Total_Paid'] > matrix['LoanAmount'] * 0.15) &
        (matrix['Advance_Payment_Count'] == 0),
        1, 0
    )

    # 5. Repayment Stress Index: monthly amortization / monthly income, as %
    matrix['Repayment_Stress_Index'] = (
        (matrix['LoanAmount'] / 12) /
        (matrix['AnnualIncome'].replace(0, np.nan) / 12)
    ) * 100

    # 6. Join demographic features
    demo_cols = [profile_id_col, 'LastName', 'FirstName', 'Occupation_Cleaned', 'occ_tier', 'Stability_Score']
    demo = df_p[[c for c in demo_cols if c in df_p.columns]].rename(
        columns={profile_id_col: 'MasterUUID'}
    ).drop_duplicates('MasterUUID')

    return matrix.merge(demo, on='MasterUUID', how='inner', suffixes=('', '_profile'))


df_final = build_master_matrix(
    df_profiles,
    df_payments_clean,
    df_loans[df_loans['MasterUUID'].notna()],
    profile_id_col,
)

print(f"✅ Master matrix: {len(df_final)} loans (matched to profiles)")
print(f"\nTarget distribution:")
dist = df_final['Target_Risk'].value_counts(normalize=True).rename({0: 'Performing', 1: 'High Risk'})
print(dist.to_string())


## 3. Phase 4 — Modeling

### 3.1 Feature engineering (MNAR-aware)

Roughly 51% of income data is missing in the source profile CSV. The naive choice — mean-imputation or `fillna(0)` — would silently destroy the predictive signal that *the act of not reporting income* is itself a risk pattern.

Instead, we:
1. Add a binary `Income_Is_Missing` flag (1 if missing, 0 if present).
2. Replace missing `Repayment_Stress_Index` with `-999` as a sentinel.

This lets the Random Forest split on missingness as a feature rather than imputing it away.

**The `prepare_features` function below is the contract.** Both training and inference must use it — any code path that builds features differently will produce silently wrong predictions.


In [ ]:
# =============================================================
# FEATURE CONTRACT — used for both training and inference
# =============================================================
FEATURE_COLUMNS = [
    'LoanAmount',
    'Stability_Score',
    'Advance_Payment_Count',
    'Income_Is_Missing',
    'Repayment_Stress_Index',
]


def prepare_features(df):
    """
    Build the model's feature matrix from a DataFrame containing:
      - LoanAmount             (numeric, PHP)
      - Stability_Score        (numeric, 1-4; from TIER_SCORE_MAP)
      - Advance_Payment_Count  (numeric, count of advance payments)
      - Repayment_Stress_Index (numeric or NaN; NaN means income unknown)

    Returns a DataFrame with columns in FEATURE_COLUMNS order.
    Must be called identically at training time and inference time.
    """
    X = pd.DataFrame(index=df.index)
    X['LoanAmount']             = df['LoanAmount']
    X['Stability_Score']        = df['Stability_Score']
    X['Advance_Payment_Count']  = df['Advance_Payment_Count']
    X['Income_Is_Missing']      = df['Repayment_Stress_Index'].isna().astype(int)
    X['Repayment_Stress_Index'] = df['Repayment_Stress_Index'].fillna(-999)
    return X[FEATURE_COLUMNS]  # enforce column order


# Build training set
X = prepare_features(df_final)
y = df_final['Target_Risk']

print(f"✅ Feature matrix: {X.shape}")
print(f"   Income missing (will use -999): {X['Income_Is_Missing'].sum()}/{len(X)} ({X['Income_Is_Missing'].mean()*100:.1f}%)")
print(f"\nFeature preview:")
print(X.head())


### 3.2 Train the model and save the handoff package

Random Forest with `class_weight='balanced'` to handle the 70/30 class imbalance, `max_depth=5` to limit overfitting on a relatively small dataset.

Outputs:
- `ttmpc_credit_risk_model.pkl` — the trained classifier
- `model_metadata.json` — full specification of features, target, and rules


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score


# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Train
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=5,
    class_weight='balanced',
    random_state=42,
)
model.fit(X_train, y_train)

# Evaluate on holdout
y_pred  = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_probs)

print("=" * 55)
print("MODEL PERFORMANCE (on 20% holdout)")
print("=" * 55)
print(f"  Accuracy : {acc*100:.2f}%")
print(f"  ROC-AUC  : {auc:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Performing (0)', 'High Risk (1)']))

# Save model
joblib.dump(model, MODEL_FILE)

# Save metadata (the developer's machine-readable contract)
metadata = {
    'model_file': MODEL_FILE.name,
    'model_type': 'sklearn.ensemble.RandomForestClassifier',
    'sklearn_version_at_training': __import__('sklearn').__version__,
    'training_date': datetime.now().isoformat(),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'test_accuracy': float(acc),
    'test_roc_auc': float(auc),
    'feature_columns_in_order': FEATURE_COLUMNS,
    'feature_order_matters': True,
    'target_meaning': {'0': 'Performing', '1': 'High Risk'},
    'target_definition': 'Target_Risk = 1 if (RemainingPrincipal > 15% of LoanAmount) AND (Advance_Payment_Count == 0)',
    'hyperparameters': {
        'n_estimators': 150,
        'max_depth': 5,
        'class_weight': 'balanced',
        'random_state': 42,
    },
    'feature_preparation_rules': {
        'LoanAmount':             'Raw loan amount in PHP (numeric).',
        'Stability_Score':        'Lookup occupation in TIER_SCORE_MAP. Default for unknown: 2.0.',
        'Advance_Payment_Count':  'Count of payments made 1-30 days BEFORE application date.',
        'Income_Is_Missing':      '1 if AnnualIncome is missing/null/zero; 0 otherwise.',
        'Repayment_Stress_Index': '(LoanAmount/12) / (AnnualIncome/12) * 100. If income missing -> -999.',
    },
}
with open(METADATA_FILE, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n💾 Model saved    : {MODEL_FILE}")
print(f"💾 Metadata saved : {METADATA_FILE}")


### 3.3 Verify — reload from disk and re-predict

A serialized model is only useful if it can be loaded cleanly in another environment. This cell reloads the `.pkl` from disk (as the developer's code will) and confirms the predictions match the in-memory model. If this passes, the handoff package is good to ship.


In [ ]:
# Reload from disk (simulates developer's environment)
loaded_model = joblib.load(MODEL_FILE)

# Re-predict on holdout
preds_fresh = loaded_model.predict(X_test)
preds_orig  = model.predict(X_test)

if (preds_fresh == preds_orig).all():
    print("✅ Serialization verified — loaded .pkl matches in-memory model.")
    print("   The handoff package is ready to deliver.")
else:
    print("🚨 MISMATCH — saved model differs from in-memory. DO NOT HAND OFF until fixed.")

print(f"\nSample predictions (first 5 holdout rows):")
preview = X_test.head(5).copy()
preview['Predicted'] = preds_fresh[:5]
preview['Actual']    = y_test.head(5).values
print(preview.to_string())


## 4. Handoff Summary

The development team receives:

| File | Purpose |
|------|---------|
| `ttmpc_credit_risk_model.pkl`  | Trained Random Forest. Load with `joblib.load()`. |
| `model_metadata.json`          | Machine-readable feature specification. |
| `feature_contract.md`          | Human-readable handoff document with examples. |
| `predict_example.py`           | Minimal working inference example. |

### Critical reminder for developers

The model expects features in **this exact order**:

```python
['LoanAmount', 'Stability_Score', 'Advance_Payment_Count',
 'Income_Is_Missing', 'Repayment_Stress_Index']
```

Passing them in any other order, or skipping the `-999` rule for missing income, produces **silently wrong predictions** (no error, just bad answers).

Use the `prepare_features()` function pattern from this notebook, or follow `predict_example.py`, as the canonical way to build inputs.
